# Bias and dark verification statistics

Contact author: Alex Broughton
<br>Date: 2026-11-13


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook measures the combined bias and dark statistics for the verification table in RTN-117.

- Clipped mean and clipped stdev per amplifier
- Unclipped stdev with defects masked, compared to the clipped stdev
- Noise that bias and dark subtraction add to a science image, as a fractional increase of the read noise
- Number of input exposures in each combined calibration and the number needed to meet the 2.5% requirement

Output PDFs are written to the current directory and copied into figures/ for the technote.


## 1.0 Set Up

In [ ]:
### Import packages and configure plotting defaults
import re

from lsst.daf.butler import Butler
from lsst.afw.cameraGeom import DetectorType
import lsst.afw.image as afwImage
import lsst.afw.math as afwMath
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

plt.rcParams.update({"font.size": 12})
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
})


butler = Butler("main")
camera = butler.get("camera", instrument="LSSTCam", collections="LSSTCam/defaults")

CALIBS = "LSSTCam/calib"
EXPOSURE = 2026010700038  # DP2 science exposure that selects the calibration epoch
DARK_TIME = 31.0  # s, dark time of a 30 s science exposure
MAX_INCREASE = 0.025  # allowed fractional growth of the read noise
MAX_RATIO = np.sqrt((1 + MAX_INCREASE) ** 2 - 1)  # allowed calibration noise / read noise

## 2.0 Load calibrations

Calibrations are selected by the validity range that contains the reference exposure. The combined bias is in electrons and the combined dark is in electrons per second. The number of input exposures is counted from the CPP_INPUT header keys.


In [ ]:
def n_inputs(calib):
    return sum(bool(re.fullmatch(r"CPP_INPUT_\d+", key)) for key in calib.getMetadata().names())


data_id = dict(instrument="LSSTCam", exposure=EXPOSURE, detector=94)
bias_ref = butler.find_dataset("bias", data_id, collections=CALIBS)
dark_ref = butler.find_dataset("dark", data_id, collections=CALIBS)
print(bias_ref.run)
print(dark_ref.run)
print(f"{n_inputs(butler.get(bias_ref))} bias inputs, {n_inputs(butler.get(dark_ref))} dark inputs")

## 3.0 Per-amplifier statistics

The defect mask is applied to the combined bias and dark of every science detector. The clipped mean and clipped stdev use 3 sigma clipping. The unclipped stdev uses every pixel outside the defect mask. Amplifiers that are entirely zero in the bias or dark are flagged. The read noise in electrons comes from the PTC.


In [ ]:
ctrl = afwMath.StatisticsControl()
ctrl.setAndMask(afwImage.Mask.getPlaneBitMask("BAD"))
flags = afwMath.MEANCLIP | afwMath.STDEVCLIP | afwMath.STDEV


def amp_stats(calib, box):
    stats = afwMath.makeStatistics(calib.maskedImage[box], flags, ctrl)
    return stats.getValue(afwMath.MEANCLIP), stats.getValue(afwMath.STDEVCLIP), stats.getValue(afwMath.STDEV)


detectors = [det for det in camera if det.getType() == DetectorType.SCIENCE]

rows = []
for det in tqdm(detectors):
    data_id = dict(instrument="LSSTCam", exposure=EXPOSURE, detector=det.getId())
    bias = butler.get("bias", data_id, collections=CALIBS)
    dark = butler.get("dark", data_id, collections=CALIBS)
    ptc = butler.get("ptc", data_id, collections=CALIBS)
    defects = butler.get("defects", data_id, collections=CALIBS)
    defects.maskPixels(bias.maskedImage)
    defects.maskPixels(dark.maskedImage)

    n_bias, n_dark = n_inputs(bias), n_inputs(dark)
    dark_exptime = dark.getMetadata()["CPP_INPUT_EXPT_0"]

    for amp in det:
        box = amp.getBBox()
        bias_mean, bias_std, bias_std_raw = amp_stats(bias, box)
        dark_mean, dark_std, dark_std_raw = amp_stats(dark, box)
        rows.append(dict(
            detector=det.getId(),
            amp=amp.getName(),
            vendor=det.getPhysicalType(),
            zero=np.all(bias.image[box].array == 0) | np.all(dark.image[box].array == 0),
            read_noise=ptc.noise[amp.getName()],
            n_bias=n_bias,
            n_dark=n_dark,
            dark_exptime=dark_exptime,
            bias_mean=bias_mean,
            bias_std=bias_std,
            bias_std_raw=bias_std_raw,
            dark_mean=dark_mean,
            dark_std=dark_std,
            dark_std_raw=dark_std_raw,
        ))

stats = pd.DataFrame(rows)

## 4.0 Added noise

The clipped stdev of each combined calibration is taken as the noise it adds to every science pixel. This is an upper bound because it also contains real fixed pattern structure. The dark stdev is scaled to the dark time of a science exposure. Each term adds to the read noise in quadrature, and the fractional increase of the read noise is compared to the 2.5% requirement.

The number of inputs needed to meet the requirement assumes the calibration noise falls as one over the square root of the number of inputs. The excess columns are the measured noise divided by the read noise expected from averaging the inputs.


In [ ]:
def noise_increase(added, read_noise):
    return np.hypot(read_noise, added) / read_noise - 1


stats["dark_noise"] = DARK_TIME * stats.dark_std
stats["added_noise"] = np.hypot(stats.bias_std, stats.dark_noise)
stats["bias_increase"] = noise_increase(stats.bias_std, stats.read_noise)
stats["dark_increase"] = noise_increase(stats.dark_noise, stats.read_noise)
stats["total_increase"] = noise_increase(stats.added_noise, stats.read_noise)

allowed_noise = MAX_RATIO * stats.read_noise
stats["n_bias_needed"] = stats.n_bias * (stats.bias_std / allowed_noise) ** 2
stats["n_dark_needed"] = stats.n_dark * (stats.dark_noise / allowed_noise) ** 2

stats["bias_excess"] = stats.bias_std * np.sqrt(stats.n_bias) / stats.read_noise
stats["dark_excess"] = stats.dark_std * stats.dark_exptime * np.sqrt(stats.n_dark) / stats.read_noise

stats["bias_std_ratio"] = stats.bias_std_raw / stats.bias_std
stats["dark_std_ratio"] = stats.dark_std_raw / stats.dark_std

print(f"Excluding {stats.zero.sum()} amplifiers that are zero in the bias or dark")
print(stats.loc[stats.zero, ["detector", "amp", "vendor"]].to_string(index=False))

good = stats[~stats.zero].dropna()
print(f"Using {len(good)} of {len(stats)} amplifiers")

## 5.0 Summary

Distribution of each statistic over the good amplifiers, followed by the medians for ITL and E2V detectors.


In [ ]:
columns = [
    "read_noise", "n_bias", "n_dark", "dark_exptime",
    "bias_mean", "bias_std", "bias_std_ratio", "bias_excess", "bias_increase", "n_bias_needed",
    "dark_mean", "dark_std", "dark_noise", "dark_std_ratio", "dark_excess", "dark_increase", "n_dark_needed",
    "added_noise", "total_increase",
]
pd.set_option("display.float_format", "{:.4g}".format)
display(good[columns].describe(percentiles=[0.05, 0.5, 0.95]).T)
display(good.groupby("vendor")[columns].median().T)

## 6.0 Number of input exposures

Compare the number of inputs in each combined calibration to the number needed to keep the read noise increase below 2.5%.


In [ ]:
for calib in ["bias", "dark"]:
    n, needed = good[f"n_{calib}"], good[f"n_{calib}_needed"]
    print(
        f"{calib}: {n.min()} to {n.max()} inputs, {needed.median():.1f} needed "
        f"(max {needed.max():.1f}), {(needed > n).sum()} amplifiers short"
    )

print(f"Input dark exposure times {good.dark_exptime.unique()} s")

## 7.0 Noise increase histograms

Fractional increase of the read noise per amplifier from the bias and from the dark. The dashed line is the 2.5% requirement.


In [ ]:
for calib in ["bias", "dark"]:
    plt.figure()
    for vendor, color in [("ITL", "darkorange"), ("E2V", "cornflowerblue")]:
        plt.hist(
            100 * good.loc[good.vendor == vendor, f"{calib}_increase"], bins=150, histtype="step", label=vendor,
            color=color, range=[0, 3], linewidth=1.5, alpha=0.75,
        )
    plt.axvline(100 * MAX_INCREASE, linestyle="--", color="k")
    plt.text(100 * MAX_INCREASE - 0.12, 0.05, "Requirement", rotation=90, transform=plt.gca().get_xaxis_transform())
    plt.xlim(0, 3)
    plt.yscale("symlog", linthresh=25)
    plt.xlabel(f"Read noise increase from {calib} (%)")
    plt.ylabel("N(bin)")
    plt.legend()
    plt.savefig(f"{calib}-noise-increase-histogram.pdf", format="pdf", bbox_inches="tight")

## 8.0 Table values

Values for the Measured column of the verification table. Means are the median over amplifiers with the scatter between amplifiers. Other values are the median over amplifiers with the worst amplifier in parentheses.


In [ ]:
med = good.median(numeric_only=True)
worst = good.max(numeric_only=True)

print(f"Bias mean               {med.bias_mean:.2e} +/- {good.bias_mean.std():.2e} e-")
print(f"Bias clipped stdev      {med.bias_std:.3f} e-, read noise +{100 * med.bias_increase:.2f}% ({100 * worst.bias_increase:.2f}%)")
print(f"Bias unclipped/clipped  {med.bias_std_ratio:.3f} ({worst.bias_std_ratio:.3f})")
print(f"Dark mean               {med.dark_mean:.2e} +/- {good.dark_mean.std():.2e} e-/s")
print(f"Dark clipped stdev      {med.dark_std:.2e} e-/s, read noise +{100 * med.dark_increase:.2f}% ({100 * worst.dark_increase:.2f}%)")
print(f"Dark unclipped/clipped  {med.dark_std_ratio:.3f} ({worst.dark_std_ratio:.3f})")
print(f"Bias and dark together  read noise +{100 * med.total_increase:.2f}% ({100 * worst.total_increase:.2f}%)")